In [144]:
import tensorflow as tf
from tensorflow.keras.layers import Input,TimeDistributed,LeakyReLU,Conv2D,Activation,MaxPooling2D,Conv2DTranspose,concatenate,Bidirectional,ConvLSTM2D,BatchNormalization,LSTM
from tensorflow.keras.layers import Conv3D, MaxPooling3D, Flatten, Dense, Dropout
from tensorflow.keras import Model,initializers, regularizers
import tensorflow.keras.backend as K


In [33]:
class Spatial_Encoder(Model):
    def __init__(self,out_channel1=16,out_channel2=32):
        super(Spatial_Encoder, self).__init__()
        self.resnet = tf.keras.applications.ResNet50V2(
                        include_top=False,
                        weights="imagenet",
                        pooling="avg")
        for layer in self.resnet.layers:
            layer.trainable = False
    
    def call(self, inputs):
        x = self.resnet(inputs)
        return x

In [93]:
def build_spatial_encoder():
    resnet = tf.keras.applications.ResNet50V2(
            include_top=False,
            weights="imagenet",
            pooling="avg")
    for layer in resnet.layers:
        layer.trainable = False
    input_tensor = Input((224,224,3), name="frame",dtype=K.floatx())
    outputs = resnet(input_tensor)
    model = Model(inputs=[input_tensor], outputs=[outputs], name="resnet50v2")
    return model

In [94]:
input = Input((224,224,3), name="ori",dtype=K.floatx())

In [95]:
model = build_spatial_encoder()
y = model(input)

In [96]:
y.shape

TensorShape([None, 2048])

In [97]:
model.summary()

Model: "resnet50v2"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
frame (InputLayer)           [(None, 224, 224, 3)]     0         
_________________________________________________________________
resnet50v2 (Model)           (None, 2048)              23564800  
Total params: 23,564,800
Trainable params: 0
Non-trainable params: 23,564,800
_________________________________________________________________


In [153]:
def build_spatial_encoder():
    resnet = tf.keras.applications.ResNet50V2(
            include_top=False,
            weights="imagenet",
            pooling="avg")
    for layer in resnet.layers:
        layer.trainable = False
    input_tensor = Input((224,224,3), name="frame",dtype=K.floatx())
    outputs = resnet(input_tensor)
    model = Model(inputs=[input_tensor], outputs=[outputs], name="resnet50v2")
    return model

def RESLSTM_Model(input_tensor):    
    spatial_encoder = build_spatial_encoder()
    x = TimeDistributed(spatial_encoder,name="spatial_encoder")(input_tensor)
    x = TimeDistributed(Dropout(0.3),name="spatial_dropout")(x)
    x = LSTM(units = 64,
            kernel_initializer='he_normal', bias_initializer='zeros',
            return_sequences=True, name="lstm_1")(x)
    x = BatchNormalization()(x)         
    x = LSTM(units = 64, 
            kernel_initializer='he_normal', bias_initializer='zeros',
            return_sequences=True, name="lstm_2")(x)
    x = BatchNormalization()(x)         
    x = LSTM(units = 32, 
            kernel_initializer='he_normal', bias_initializer='zeros',
            return_sequences=True, name="lstm_3")(x) 
    x = BatchNormalization()(x)     
    x = TimeDistributed(Dense(LABEL_NUM))(x)
    outputs = Activation("softmax")(x)
    model = Model(inputs=[input_tensor], outputs=[outputs], name="resnet_LSTM")
    return model

In [154]:
input = Input((16,224,224,3), name="ori",dtype=K.floatx())

In [155]:
LABEL_NUM = 3
model = RESLSTM_Model(input)

In [156]:
model.summary()

Model: "resnet_LSTM"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
ori (InputLayer)             [(None, 16, 224, 224, 3)] 0         
_________________________________________________________________
spatial_encoder (TimeDistrib (None, 16, 2048)          23564800  
_________________________________________________________________
spatial_dropout (TimeDistrib (None, 16, 2048)          0         
_________________________________________________________________
lstm_1 (LSTM)                (None, 16, 64)            540928    
_________________________________________________________________
batch_normalization_53 (Batc (None, 16, 64)            256       
_________________________________________________________________
lstm_2 (LSTM)                (None, 16, 64)            33024     
_________________________________________________________________
batch_normalization_54 (Batc (None, 16, 64)            

In [145]:
def TimeDistributed_Conv2D_Block(input_tensor, out_channels):
    # first layer
    x = TimeDistributed(Conv2D(filters=out_channels, kernel_size=[3,3], padding="same",
                        kernel_initializer="he_normal",
                        bias_initializer="zeros"))(input_tensor)
    x = TimeDistributed(BatchNormalization())(x)
    x = TimeDistributed(LeakyReLU())(x)
    #x = TimeDistributed(MaxPooling2D(pool_size=[2,2], strides=[2,2], padding="same"))(x)
    
    # second layer
    x = TimeDistributed(Conv2D(filters=out_channels, kernel_size=[3,3], padding="same",
                        kernel_initializer="he_normal",
                        bias_initializer="zeros"))(x)
    x = TimeDistributed(BatchNormalization())(x)
    x = TimeDistributed(LeakyReLU())(x)
    x = TimeDistributed(MaxPooling2D(pool_size=[2,2], strides=[2,2], padding="same"))(x)   
    return x
    
def CNNLSTM_Model(input_tensor,n_filters=16):        
    x = TimeDistributed_Conv2D_Block(input_tensor,n_filters)
    x = TimeDistributed_Conv2D_Block(x,n_filters)
    x = TimeDistributed_Conv2D_Block(x,n_filters*2)
    x = TimeDistributed_Conv2D_Block(x,n_filters*2)
    x = TimeDistributed_Conv2D_Block(x,n_filters*4)   

    x = TimeDistributed(Flatten())(x)

    x = TimeDistributed(Dense(128))(x)
    x = TimeDistributed(LeakyReLU())(x)
    x = BatchNormalization()(x)
    x = TimeDistributed(Dropout(0.3))(x)
    
    x = LSTM(units = 64,
            kernel_initializer='he_normal', bias_initializer='zeros',
            return_sequences=True)(x)
    x = BatchNormalization()(x)         
    x = LSTM(units = 64, 
            kernel_initializer='he_normal', bias_initializer='zeros',
            return_sequences=True)(x)
    x = BatchNormalization()(x)         
    x = LSTM(units = 32, 
            kernel_initializer='he_normal', bias_initializer='zeros',
            return_sequences=True)(x) 
    x = BatchNormalization()(x)     
    x = TimeDistributed(Dense(LABEL_NUM))(x)
    #x = Dense(LABEL_NUM)(x)
    outputs = Activation("softmax")(x)
    model = Model(inputs=[input_tensor], outputs=[outputs], name="CNN-LSTM")
    return model

In [146]:
input = Input((16,224,224,3), name="ori",dtype=K.floatx())
model = CNNLSTM_Model(input)
model.summary()

Model: "CNN-LSTM"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
ori (InputLayer)             [(None, 16, 224, 224, 3)] 0         
_________________________________________________________________
time_distributed_37 (TimeDis (None, 16, 224, 224, 16)  448       
_________________________________________________________________
time_distributed_38 (TimeDis (None, 16, 224, 224, 16)  64        
_________________________________________________________________
time_distributed_39 (TimeDis (None, 16, 224, 224, 16)  0         
_________________________________________________________________
time_distributed_40 (TimeDis (None, 16, 224, 224, 16)  2320      
_________________________________________________________________
time_distributed_41 (TimeDis (None, 16, 224, 224, 16)  64        
_________________________________________________________________
time_distributed_42 (TimeDis (None, 16, 224, 224, 16)  0  